# Bronze Ingestion: Planetary Computer STAC Metadata

This notebook demonstrates a lightweight, parameterized bronze ingestion pattern for Fabric CI/CD demos.

Flow:
1. Define parameters (collection, location, radius, dates).
2. Query Microsoft Planetary Computer STAC API.
3. Write item metadata into a Lakehouse bronze Delta table.

In [ ]:
# Parameters (edit these values directly, or replace with parameter cell tags in Fabric pipelines)
COLLECTION = "sentinel-2-l2a"
LATITUDE = 49.2193
LONGITUDE = -122.5984
RADIUS_KM = 20
START_DATE = "2024-06-01"
END_DATE = "2024-09-30"
MAX_ITEMS = 150
TARGET_TABLE = "bronze_satellite_stac_items"

print("Parameters loaded")
print({
    "collection": COLLECTION,
    "lat": LATITUDE,
    "lon": LONGITUDE,
    "radius_km": RADIUS_KM,
    "start_date": START_DATE,
    "end_date": END_DATE,
    "max_items": MAX_ITEMS,
    "target_table": TARGET_TABLE
})

In [ ]:
import math
import requests

STAC_SEARCH_URL = "https://planetarycomputer.microsoft.com/api/stac/v1/search"

def radius_to_bbox(lat, lon, radius_km):
    # Approximate conversion from km radius to lat/lon degree offsets.
    lat_delta = radius_km / 111.32
    cos_lat = math.cos(math.radians(lat))
    if abs(cos_lat) < 1e-8:
        lon_delta = 180.0
    else:
        lon_delta = radius_km / (111.32 * cos_lat)
    min_lon = lon - lon_delta
    min_lat = lat - lat_delta
    max_lon = lon + lon_delta
    max_lat = lat + lat_delta
    return [min_lon, min_lat, max_lon, max_lat]

bbox = radius_to_bbox(LATITUDE, LONGITUDE, RADIUS_KM)
datetime_range = f"{START_DATE}T00:00:00Z/{END_DATE}T23:59:59Z"

payload = {
    "collections": [COLLECTION],
    "bbox": bbox,
    "datetime": datetime_range,
    "limit": MAX_ITEMS
}

response = requests.post(STAC_SEARCH_URL, json=payload, timeout=60)
response.raise_for_status()
data = response.json()
features = data.get("features", [])

print(f"STAC items returned: {len(features)}")
if len(features) > 0:
    print("Sample item id:", features[0].get("id"))

In [ ]:
from pyspark.sql import Row
from pyspark.sql.functions import current_timestamp

rows = []
for f in features:
    props = f.get("properties", {})
    bbox_vals = f.get("bbox", [None, None, None, None])
    assets = f.get("assets", {})

    rows.append(Row(
        item_id=str(f.get("id")) if f.get("id") is not None else None,
        collection=str(f.get("collection")) if f.get("collection") is not None else COLLECTION,
        datetime_utc=str(props.get("datetime")) if props.get("datetime") is not None else None,
        platform=str(props.get("platform")) if props.get("platform") is not None else None,
        eo_cloud_cover=float(props.get("eo:cloud_cover")) if props.get("eo:cloud_cover") is not None else None,
        bbox_minx=float(bbox_vals[0]) if len(bbox_vals) > 0 and bbox_vals[0] is not None else None,
        bbox_miny=float(bbox_vals[1]) if len(bbox_vals) > 1 and bbox_vals[1] is not None else None,
        bbox_maxx=float(bbox_vals[2]) if len(bbox_vals) > 2 and bbox_vals[2] is not None else None,
        bbox_maxy=float(bbox_vals[3]) if len(bbox_vals) > 3 and bbox_vals[3] is not None else None,
        asset_count=int(len(assets)),
        source_api="planetary_computer_stac",
        query_lat=float(LATITUDE),
        query_lon=float(LONGITUDE),
        query_radius_km=float(RADIUS_KM),
        query_start_date=START_DATE,
        query_end_date=END_DATE
    ))

if len(rows) == 0:
    raise ValueError("No items returned for the selected query parameters.")

df = spark.createDataFrame(rows).withColumn("ingestion_ts", current_timestamp())

(
    df.write
      .format("delta")
      .mode("append")
      .saveAsTable(TARGET_TABLE)
)

print(f"Wrote {df.count()} rows to table: {TARGET_TABLE}")

In [ ]:
# Quick verification
spark.sql(f"SELECT collection, COUNT(*) AS rows FROM {TARGET_TABLE} GROUP BY collection ORDER BY rows DESC").show(truncate=False)
spark.sql(f"SELECT * FROM {TARGET_TABLE} ORDER BY ingestion_ts DESC LIMIT 10").show(truncate=False)